# TCN + FiLM — Sequential Pose Correction

This notebook trains a **Temporal Convolutional Network (TCN)** with **dilated causal convolutions** for **sequence → vector regression**.

- **Input**: pose window `(T, 12 * n_coords)`
- **Conditioning**: exercise label `y_class` injected via **FiLM** at each residual block
- **Output**: final joint displacement vector `y` of shape `(12 * n_coords,)`

TCNs are fast, parallelizable, and capture multi-scale temporal patterns via dilations.


In [ ]:
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split

tf.random.set_seed(0)
np.random.seed(0)
print("tf:", tf.__version__)


## Load displacement training data

Expected `training_data_displacement.npz` keys:

- `X`: flattened pose window + class features
- `y`: final displacement vector
- `y_class`: integer class id
- `class_names`, plus metadata keys like `n_landmarks`, `n_coords`, `sequence_length`


In [ ]:
data_path = Path("../../Data/output_displacement/training_data_displacement.npz")
metadata_path = Path("../../Data/output_displacement/training_data_displacement_metadata.json")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("Dataset metadata:")
for k in (
    "n_samples",
    "sequence_length",
    "n_landmarks",
    "X_pose_flat_dim",
    "X_class_dim",
    "y_final_disp_flat_dim",
    "class_encoding",
):
    if k in metadata:
        print(f"  {k}: {metadata[k]}")
print(f"  n_coords: {int(metadata.get('n_coords', 3))}  (older JSON without key defaults to 3)")

bundle = np.load(data_path, allow_pickle=True)
print(f"\n.npz keys: {list(bundle.keys())}")

X = bundle["X"].astype(np.float32)
y = bundle["y"].astype(np.float32)
y_class = bundle["y_class"].astype(np.int32)
class_names = bundle["class_names"]

n_classes = int(len(class_names))
print(f"\nX: {X.shape}, y: {y.shape}, y_class: {y_class.shape}, classes: {n_classes}")


## Preprocess (pose-only normalization + standardized targets)

We match `LSTM_embedding_pose_correction.ipynb`:

- Reshape pose to `(N, T, 12 * n_coords)`
- Z-score **pose only** over the full dataset (train+test)
- Standardize `y` per output dimension
- Keep `y_class` as integer ids for FiLM conditioning


In [ ]:
N = int(X.shape[0])
T = int(metadata["sequence_length"])
pose_flat_dim = int(metadata["X_pose_flat_dim"])

n_coords = int(metadata.get("n_coords", 3))
n_landmarks = int(metadata["n_landmarks"])
pose_feats_per_step = n_landmarks * n_coords

X_pose_flat = X[:, :pose_flat_dim]
X_pose_seq = X_pose_flat.reshape(N, T, -1)
if int(X_pose_seq.shape[2]) != int(pose_feats_per_step):
    raise ValueError(
        f"Expected {pose_feats_per_step} pose features per timestep "
        f"(n_landmarks={n_landmarks} * n_coords={n_coords}), got {X_pose_seq.shape[2]}"
    )

X_pose_seq = np.nan_to_num(X_pose_seq, nan=0.0, posinf=0.0, neginf=0.0)
y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

X_mean = np.mean(X_pose_seq, axis=(0, 1), keepdims=True)
X_std = np.std(X_pose_seq, axis=(0, 1), keepdims=True) + 1e-8
X_pose_norm = (X_pose_seq - X_mean) / X_std

y_mean = np.mean(y, axis=0, keepdims=True)
y_std = np.std(y, axis=0, keepdims=True) + 1e-8
y_standardized = (y - y_mean) / y_std

out_dim = int(y_standardized.shape[1])
print(f"X_pose_norm: {X_pose_norm.shape}, y_standardized: {y_standardized.shape}, n_classes: {n_classes}")


## Train / test split (stratified by workout class)


In [ ]:
X_pose_train, X_pose_test, y_train, y_test, y_class_train, y_class_test = train_test_split(
    X_pose_norm,
    y_standardized,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class,
)

print(f"train: {X_pose_train.shape[0]}, test: {X_pose_test.shape[0]}")
print(f"pose: {X_pose_train.shape[1:]} → target dim {y_train.shape[1]}")


## TCN with FiLM conditioning

We build a residual TCN stack:

- **Causal** `Conv1D(padding="causal")`
- **Dilations** to grow the receptive field: e.g. `1, 2, 4, 8, ...`
- **FiLM** per block: from class embedding → `(gamma, beta)` with shape `(channels,)`, then apply

\[ \mathrm{FiLM}(h; c) = \gamma(c) \odot h + \beta(c) \]

where `h` is the block activation `(B, T, channels)`.


In [ ]:
def make_tcn_film_model(
    n_timesteps: int,
    n_pose: int,
    n_classes: int,
    out_dim: int,
    *,
    channels: int = 128,
    n_blocks: int = 8,
    kernel_size: int = 3,
    dropout: float = 0.1,
    class_emb_dim: int = 32,
    film_hidden: int = 128,
):
    pose_in = layers.Input(shape=(n_timesteps, n_pose), name="pose")
    class_id = layers.Input(shape=(1,), dtype="int32", name="class_id")

    # Class embedding -> conditioning vector.
    c = layers.Embedding(n_classes, class_emb_dim, name="class_embedding")(class_id)
    c = layers.Reshape((class_emb_dim,), name="class_emb_flat")(c)
    c = layers.Dense(film_hidden, activation="swish", name="cond_mlp_1")(c)
    c = layers.Dense(film_hidden, activation="swish", name="cond_mlp_2")(c)

    # Project pose to channels.
    x = layers.Conv1D(channels, kernel_size=1, padding="same", name="in_proj")(pose_in)

    for i in range(n_blocks):
        dilation = 2**i
        residual = x

        h = layers.LayerNormalization(name=f"ln_{i}")(x)
        h = layers.Activation("swish", name=f"act_{i}_pre")(h)
        h = layers.Conv1D(
            channels,
            kernel_size=kernel_size,
            dilation_rate=dilation,
            padding="causal",
            name=f"tcn_conv_{i}",
        )(h)
        h = layers.Dropout(dropout, name=f"drop_{i}")(h)

        # FiLM parameters for this block.
        # Avoid Lambda(tf.split(...)) so the model can be deserialized cleanly in Keras 3.
        gamma = layers.Dense(
            channels,
            kernel_initializer="zeros",
            bias_initializer="ones",
            name=f"film_gamma_{i}",
        )(c)
        beta = layers.Dense(
            channels,
            kernel_initializer="zeros",
            bias_initializer="zeros",
            name=f"film_beta_{i}",
        )(c)

        # Broadcast to time axis.
        gamma = layers.Reshape((1, channels), name=f"film_gamma_reshape_{i}")(gamma)
        beta = layers.Reshape((1, channels), name=f"film_beta_reshape_{i}")(beta)

        h = layers.Multiply(name=f"film_mul_{i}")([h, gamma])
        h = layers.Add(name=f"film_add_{i}")([h, beta])

        # Residual connection.
        x = layers.Add(name=f"res_add_{i}")([residual, h])

    x = layers.LayerNormalization(name="post_ln")(x)
    x = layers.Activation("swish", name="post_act")(x)
    x = layers.GlobalAveragePooling1D(name="gap")(x)

    x = layers.Dense(128, activation="swish", name="head_1")(x)
    x = layers.Dropout(0.2, name="head_drop_1")(x)
    x = layers.Dense(64, activation="swish", name="head_2")(x)
    x = layers.Dropout(0.2, name="head_drop_2")(x)

    disp = layers.Dense(out_dim, name="disp")(x)

    model = keras.Model(inputs=[pose_in, class_id], outputs={"disp": disp}, name="tcn_film_pose_correction")
    return model

n_timesteps = int(X_pose_train.shape[1])
n_pose = int(X_pose_train.shape[2])

model = make_tcn_film_model(
    n_timesteps=n_timesteps,
    n_pose=n_pose,
    n_classes=n_classes,
    out_dim=out_dim,
    channels=128,
    n_blocks=7,       # receptive field covers T=15 easily with k=3
    kernel_size=3,
    dropout=0.1,
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-3),
    loss={"disp": "mse"},
    metrics={"disp": [keras.metrics.MeanAbsoluteError(name="mae")]},
)

model


## Training

We use `tf.data` for efficient batching and parallel loading.


In [ ]:
def make_ds(X_pose: np.ndarray, y: np.ndarray, y_class: np.ndarray, *, batch_size: int, training: bool):
    ds = tf.data.Dataset.from_tensor_slices(
        ((X_pose.astype(np.float32), y_class.astype(np.int32).reshape(-1, 1)), y.astype(np.float32))
    )
    if training:
        ds = ds.shuffle(min(int(len(X_pose)), 50_000), seed=0, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

# Match other pose_correction notebooks: save into ./models
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

batch_size = 512
train_ds = make_ds(X_pose_train, y_train, y_class_train, batch_size=batch_size, training=True)
val_ds = make_ds(X_pose_test, y_test, y_class_test, batch_size=batch_size, training=False)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(models_dir / "tcn_film_pose_correction_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=4, factor=0.5, min_lr=2e-5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200,
    callbacks=callbacks,
)


## Evaluation

We report metrics both in standardized space and after un-standardizing back to displacement units.


In [ ]:
def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true, axis=0, keepdims=True)) ** 2))
    return 1.0 - ss_res / (ss_tot + 1e-12)

pred = model.predict((X_pose_test, y_class_test.reshape(-1, 1)), batch_size=1024, verbose=0)
y_pred_std = pred["disp"].astype(np.float32)

mse_std = float(np.mean((y_test - y_pred_std) ** 2))
mae_std = float(np.mean(np.abs(y_test - y_pred_std)))
r2_std = r2_score(y_test, y_pred_std)

# Un-standardize.
y_test_raw = y_test * y_std + y_mean
y_pred_raw = y_pred_std * y_std + y_mean

mse_raw = float(np.mean((y_test_raw - y_pred_raw) ** 2))
mae_raw = float(np.mean(np.abs(y_test_raw - y_pred_raw)))
r2_raw = r2_score(y_test_raw, y_pred_raw)

print("Standardized:")
print("  MSE:", mse_std)
print("  MAE:", mae_std)
print("  R2 :", r2_std)
print("\nRaw displacement units:")
print("  MSE:", mse_raw)
print("  MAE:", mae_raw)
print("  R2 :", r2_raw)


## Save model + preprocessing stats

We save:

- Keras model directory (for `pose_correction_prediction_visualizer.py`-style loading)
- `*_preprocess_stats.npz` with `X_mean`, `X_std`, `y_mean`, `y_std`


In [ ]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

final_path = models_dir / "tcn_film_pose_correction_final.keras"
model.save(final_path)

stats_path = models_dir / "tcn_film_preprocess_stats.npz"
np.savez(
    stats_path,
    X_mean=X_mean.astype(np.float32),
    X_std=X_std.astype(np.float32),
    y_mean=y_mean.astype(np.float32),
    y_std=y_std.astype(np.float32),
    n_timesteps=np.asarray([n_timesteps], dtype=np.int32),
    n_pose=np.asarray([n_pose], dtype=np.int32),
    n_classes=np.asarray([n_classes], dtype=np.int32),
)

print("saved final model:", final_path)
print("saved stats:", stats_path)
print("saved best checkpoint:", models_dir / "tcn_film_pose_correction_best.keras")
